In [28]:
import pandas as pd
from sklearn.model_selection import train_test_split
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.optim as optim

In [29]:
df = pd.read_csv(r"fmnist_small.csv")
df.head()

,label,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,pixel9,...,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783,pixel784
0,9,0,0,0,0,0,0,0,0,0,...,0,7,0,50,205,196,213,165,0,0
1,7,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,1,0,0,0,...,142,142,142,21,0,3,0,0,0,0
3,8,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,8,0,0,0,0,0,0,0,0,0,...,213,203,174,151,188,10,0,0,0,0


In [30]:
x = df.iloc[:, 1:]
y = df.iloc[:,0]

In [31]:
x_train, x_test, y_train, y_test = train_test_split(x,y,test_size=0.2,random_state=42)

In [32]:
x_train = x_train/255.0
x_test = x_test/255.0

In [41]:
class CustomDataset(Dataset):

  def __init__(self, features, labels):

    
    self.features = torch.tensor(features.values, dtype=torch.float32)
    self.labels = torch.tensor(labels.values, dtype=torch.long)

  def __len__(self):
    return len(self.features)

  def __getitem__(self, index):

    image = self.features[index].view(1, 28, 28)

    label = self.labels[index]

    return image, label

In [42]:
train_dataset = CustomDataset(x_train, y_train)
test_dataset = CustomDataset(x_test, y_test)

In [43]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, pin_memory=True)

In [44]:
class MyNN(nn.Module):
    def __init__(self, input_features):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(input_features, 32, kernel_size=3, padding='same'),
            nn.ReLU(),
            nn.BatchNorm2d(32),
            nn.MaxPool2d(kernel_size=2, stride=2),

            nn.Conv2d(32, 64, kernel_size=3, padding='same'),
            nn.ReLU(),
            nn.BatchNorm2d(64),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.LazyLinear(128),
            nn.ReLU(),
            nn.Dropout(p=0.4),

            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(p=0.4),

            nn.Linear(64, 10)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)

        return x

In [45]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

device(type='cuda')

In [ ]:
learning_rate = 0.1
epoch = 100

In [ ]:
model = MyNN(1)

model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=learning_rate, weight_decay=1e-4)

In [48]:
# training loop

for epoch in range(epoch):

  total_epoch_loss = 0

  for batch_features, batch_labels in train_loader:

    # move data to gpu
    batch_features, batch_labels = batch_features.to(device), batch_labels.to(device)

    # forward pass
    outputs = model(batch_features)

    # calculate loss
    loss = criterion(outputs, batch_labels)

    # back pass
    optimizer.zero_grad()
    loss.backward()

    # update grads
    optimizer.step()

    total_epoch_loss = total_epoch_loss + loss.item()

  avg_loss = total_epoch_loss/len(train_loader)
  print(f'Epoch: {epoch + 1} , Loss: {avg_loss}')


Epoch: 1 , Loss: 1.0478915067513783
Epoch: 2 , Loss: 0.6914417723814646
Epoch: 3 , Loss: 0.6031992262601853
Epoch: 4 , Loss: 0.5275222866733869
Epoch: 5 , Loss: 0.4753133218487104
Epoch: 6 , Loss: 0.4372077213227749
Epoch: 7 , Loss: 0.3917279028892517
Epoch: 8 , Loss: 0.3882278084754944
Epoch: 9 , Loss: 0.34374010011553763
Epoch: 10 , Loss: 0.32661734879016874
Epoch: 11 , Loss: 0.29412333432585
Epoch: 12 , Loss: 0.297935022637248
Epoch: 13 , Loss: 0.2604778147737185
Epoch: 14 , Loss: 0.2708386812110742
Epoch: 15 , Loss: 0.22619353696703912
Epoch: 16 , Loss: 0.2240069235364596
Epoch: 17 , Loss: 0.20517604071646928
Epoch: 18 , Loss: 0.20455202642828227
Epoch: 19 , Loss: 0.20181490781872222
Epoch: 20 , Loss: 0.17746815179785091
Epoch: 21 , Loss: 0.16218759170733393
Epoch: 22 , Loss: 0.17993322539453704
Epoch: 23 , Loss: 0.16896092186992367
Epoch: 24 , Loss: 0.16204358260768156
Epoch: 25 , Loss: 0.12208504664401214
Epoch: 26 , Loss: 0.12635625740260972
Epoch: 27 , Loss: 0.11590308898904671

In [ ]:
model.eval()

MyNN(
  (features): Sequential(
    (0): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=same)
    (1): ReLU()
    (2): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (4): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=same)
    (5): ReLU()
    (6): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (7): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (classifier): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=3136, out_features=128, bias=True)
    (2): ReLU()
    (3): Dropout(p=0.4, inplace=False)
    (4): Linear(in_features=128, out_features=64, bias=True)
    (5): ReLU()
    (6): Dropout(p=0.4, inplace=False)
    (7): Linear(in_features=64, out_features=10, bias=True)
  )
)

In [53]:
total = 0
correct = 0

with torch.no_grad():
    for features, labels in train_loader:
        features, labels = features.to(device), labels.to(device)

        y_pred = model(features)
        _, predicted = torch.max(y_pred, 1)
        total+=labels.size(0)
        correct+=(predicted==labels).sum().item()
        
accuracy = correct/total
print(accuracy)        

0.9991666666666666


In [54]:
total = 0
correct = 0

with torch.no_grad():
    for features, labels in test_loader:
        features, labels = features.to(device), labels.to(device)

        y_pred = model(features)
        _, predicted = torch.max(y_pred, 1)
        total+=labels.size(0)
        correct+=(predicted==labels).sum().item()
        
accuracy = correct/total
print(accuracy)        

0.8691666666666666
